## 1. Import & load libraries

In [ ]:
%pip cache purge
%pip install -r ../requirements.txt


In [ ]:
# import mne
# from mne.decoding import CSP
# from mne.preprocessing import ICA

# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# from sklearn.svm import SVC
# from sklearn.model_selection import cross_val_score

# import numpy as np
# import pandas as pd

# import os
# import warnings
# import json
# import time
# import sys

# import pickle

# # from tqdm import tqdm

# from typing import List

# # sys.path.append(os.path.abspath(".."))
# # from utils.pca import analyze_eeg_pca


In [ ]:
import gc  # Garbage Collector
gc.enable()
gc.get_stats()


## 2. Pipeline

In [ ]:
# models_dir = '../models'

# # 1. Create the pipeline with dimensionality reduction and classification
# def create_processing_pipeline(n_components=0.96):
#     pipeline = Pipeline([
#         ('dimension_reduction', PCA(n_components=n_components)),
#         ('classifier', SVC(kernel='rbf'))
#     ])
#     return pipeline

# # 2. Load the preprocessed data
# X_train = np.load(os.path.join(models_dir, 'X_preprocessed.npy'))
# y_train = np.load(os.path.join(models_dir, 'y_labels.npy'))

# # 3. Create and train the pipeline
# pipeline = create_processing_pipeline()
# pipeline.fit(X_train, y_train)

# # 4. Evaluate using cross-validation
# scores = cross_val_score(pipeline, X_train, y_train, cv=5)
# print(f"Cross-validation scores: {scores}")
# print(f"Mean CV score: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

# # 5. Save the trained pipeline
# pipeline_info = {
#     'pipeline_params': {
#         'pca_n_components': float(pipeline.named_steps['dimension_reduction'].n_components_),
#         'n_components_selected': int(pipeline.named_steps['dimension_reduction'].n_components_),
#         'explained_variance_ratio': [float(x) for x in pipeline.named_steps['dimension_reduction'].explained_variance_ratio_],
#         'classifier_params': {k: str(v) if isinstance(v, (np.int64, np.float64)) else v 
#                             for k, v in pipeline.named_steps['classifier'].get_params().items()}
#     }
# }

# # Save pipeline information to JSON
# with open(os.path.join(models_dir, 'pipeline_info.json'), 'w') as f:
#     json.dump(pipeline_info, f, indent=4)

# # Save the complete pipeline using pickle
# with open(os.path.join(models_dir, 'trained_pipeline.pkl'), 'wb') as f:
#     pickle.dump(pipeline, f)

# # 6. Simulate real-time prediction (playback)
# def simulate_real_time_prediction(pipeline, X, y, chunk_size=10):
#     predictions = []
#     true_labels = []
    
#     for i in range(0, len(X), chunk_size):
#         # Get a chunk of data
#         X_chunk = X[i:i + chunk_size]
#         y_chunk = y[i:i + chunk_size]
        
#         # Make predictions
#         y_pred = pipeline.predict(X_chunk)
        
#         predictions.extend(y_pred.tolist())  # Convert to list
#         true_labels.extend(y_chunk.tolist())  # Convert to list
        
#         # Calculate current accuracy
#         current_acc = np.mean(np.array(predictions) == np.array(true_labels))
#         print(f"Processed {i+len(X_chunk)}/{len(X)} samples. Current accuracy: {current_acc:.3f}")
        
#         # Simulate real-time delay
#         time.sleep(0.1)
    
#     return predictions, true_labels

# # Test the real-time simulation
# print("\nTesting real-time prediction simulation:")
# predictions, true_labels = simulate_real_time_prediction(pipeline, X_train[:100], y_train[:100])
# final_accuracy = np.mean(np.array(predictions) == np.array(true_labels))
# print(f"\nFinal accuracy on simulation: {final_accuracy:.3f}")


In [ ]:
# # Load preprocessing info to get the channel information
# with open(os.path.join(models_dir, 'preprocessing_info.json'), 'r') as f:
#     preprocess_info = json.load(f)

# # Get dimensions
# n_epochs = len(X_train)
# n_channels = len(preprocess_info['channels'])
# n_times = X_train.shape[1] // n_channels

# print(f"Data dimensions:")
# print(f"Number of epochs: {n_epochs}")
# print(f"Number of channels: {n_channels}")
# print(f"Time points: {n_times}")

# # Reshape to (n_epochs, n_channels, n_times)
# X_train_3d = X_train.reshape(n_epochs, n_channels, n_times)

# def create_csp_pipeline(n_components=4):
#     pipeline = Pipeline([
#         ('csp', CSP(n_components=n_components, reg=None, log=True)),
#         ('classifier', SVC(kernel='rbf'))
#     ])
#     return pipeline

# def create_pca_pipeline(n_components=0.95):
#     pipeline = Pipeline([
#         ('dimension_reduction', PCA(n_components=n_components)),
#         ('classifier', SVC(kernel='rbf'))
#     ])
#     return pipeline

# # Create pipelines
# pipelines = {
#     'PCA': create_pca_pipeline(),
#     'CSP': create_csp_pipeline()
# }

# # Evaluate each pipeline
# for name, pipe in pipelines.items():
#     if name == 'CSP':
#         # Use 3D data for CSP
#         scores = cross_val_score(pipe, X_train_3d, y_train, cv=5)
#     else:
#         # Use flattened data for PCA
#         scores = cross_val_score(pipe, X_train, y_train, cv=5)
    
#     print(f"\n🧠 {name} Pipeline:")
#     print(f"Cross-validation scores: {scores}")
#     print(f"Mean CV score: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")


In [ ]:
import numpy as np
import os
import json
import pickle
import time
import datetime
from tqdm import tqdm
from joblib import dump, load
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.base import BaseEstimator, ClassifierMixin
from mne.decoding import CSP
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy import signal

# Directory to save models
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

class EnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom Ensemble classifier that combines CSP and non-CSP models for improved performance
    """
    def __init__(self, standard_pipeline=None, csp_pipeline=None, voting='soft'):
        self.standard_pipeline = standard_pipeline
        self.csp_pipeline = csp_pipeline
        self.voting = voting
        self.is_fitted_ = False
        self.classes_ = None
        
    def fit(self, X, y, X_3d=None):
        if X_3d is None:
            # Try to automatically convert to 3D format for CSP
            try:
                n_samples = X.shape[0]
                n_channels = 64  # Assume 64 channels by default
                n_times = X.shape[1] // n_channels
                X_3d = X.reshape(n_samples, n_channels, n_times).astype(np.float64)
            except:
                raise ValueError("Could not convert to 3D format automatically. Please provide X_3d explicitly.")
        
        # Train standard pipeline
        if self.standard_pipeline is not None:
            self.standard_pipeline.fit(X, y)
            
        # Train CSP pipeline
        if self.csp_pipeline is not None and X_3d is not None:
            self.csp_pipeline.fit(X_3d, y)
        
        # Save classes
        self.classes_ = np.unique(y)
        self.is_fitted_ = True
        return self
    
    def predict_proba(self, X, X_3d=None):
        if not self.is_fitted_:
            raise ValueError("Model must be trained before prediction.")
            
        if X_3d is None:
            # Try to automatically convert to 3D format for CSP
            try:
                n_samples = X.shape[0]
                n_channels = 64  # Assume 64 channels by default
                n_times = X.shape[1] // n_channels
                X_3d = X.reshape(n_samples, n_channels, n_times).astype(np.float64)
            except:
                raise ValueError("Could not convert to 3D format automatically. Please provide X_3d explicitly.")
        
        # Calculate probabilities from both pipelines
        probs = []
        
        if self.standard_pipeline is not None:
            std_proba = self.standard_pipeline.predict_proba(X)
            probs.append(std_proba)
            
        if self.csp_pipeline is not None:
            csp_proba = self.csp_pipeline.predict_proba(X_3d)
            probs.append(csp_proba)
        
        # Average probabilities
        if len(probs) > 1:
            final_proba = np.mean(probs, axis=0)
        else:
            final_proba = probs[0]
            
        return final_proba
    
    def predict(self, X, X_3d=None):
        proba = self.predict_proba(X, X_3d)
        return self.classes_[np.argmax(proba, axis=1)]

def augment_eeg_data(X, y, X_3d=None, noise_level=0.1, shift_range=10, n_augmentations=1):
    """
    Performs data augmentation for EEG using noise, shift and SMOTE
    """
    print("🔄 Augmenting EEG data...")
    n_samples, n_features = X.shape
    
    # 1. Apply SMOTE first to balance classes
    print("  • Applying SMOTE for class balancing")
    smote = SMOTE(random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)
    
    # 2. Generate samples with noise
    X_augmented = [X_smote]
    y_augmented = [y_smote]
    
    # 3. Add Gaussian noise
    print(f"  • Adding Gaussian noise (level: {noise_level})")
    for i in range(n_augmentations):
        noise = np.random.normal(0, noise_level, X_smote.shape)
        X_augmented.append(X_smote + noise)
        y_augmented.append(y_smote)
    
    # 4. If we have 3D data, augment it as well
    if X_3d is not None:
        X_3d_augmented = []
        n_samples_aug = len(y_smote)
        
        # Reshape X_smote for 3D
        try:
            n_channels = X_3d.shape[1]
            n_times = X_3d.shape[2]
            X_smote_3d = X_smote.reshape(n_samples_aug, n_channels, n_times)
            X_3d_augmented = [X_smote_3d]
            
            # Apply temporal shift
            print(f"  • Applying temporal shift (range: {shift_range})")
            for i in range(n_augmentations):
                X_shifted = np.zeros_like(X_smote_3d)
                shifts = np.random.randint(-shift_range, shift_range, size=n_samples_aug)
                
                for j, shift in enumerate(shifts):
                    if shift > 0:
                        X_shifted[j, :, shift:] = X_smote_3d[j, :, :-shift]
                        X_shifted[j, :, :shift] = X_smote_3d[j, :, :shift]  # Repeat first values
                    elif shift < 0:
                        X_shifted[j, :, :shift] = X_smote_3d[j, :, -shift:]
                        X_shifted[j, :, shift:] = X_smote_3d[j, :, shift:]  # Repeat last values
                    else:
                        X_shifted[j] = X_smote_3d[j]
                
                X_3d_augmented.append(X_shifted)
            
            # Concatenate augmented 3D data
            X_3d_final = np.vstack(X_3d_augmented)
        except Exception as e:
            print(f"⚠️ Error augmenting 3D data: {str(e)}")
            X_3d_final = None
    else:
        X_3d_final = None
    
    # Concatenate all augmented data
    X_final = np.vstack(X_augmented)
    y_final = np.hstack(y_augmented)
    
    print(f"✅ Data augmentation completed: {n_samples} → {len(y_final)} samples")
    return X_final, y_final, X_3d_final

def extract_eeg_features(X, X_3d=None, sfreq=160, simplified=True):
    """
    Extracts additional features from EEG signals, with a simplified option
    """
    print("📊 Extracting advanced EEG features...")
    
    # Frequency domain features
    if X_3d is not None and not simplified:
        n_samples, n_channels, n_times = X_3d.shape
        
        # Define frequency bands
        bands = {
            'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 13),
            'beta': (13, 30),  # Combined beta band
            'gamma': (30, 45)
        }
        
        # Calculate PSD (Power Spectral Density)
        print("  • Calculating power in frequency bands")
        features = []
        
        for i in tqdm(range(n_samples), desc="Processing samples"):
            sample_features = []
            
            for ch in range(n_channels):
                signal_channel = X_3d[i, ch, :]
                
                # Apply Hanning window
                windowed_signal = signal.windows.hann(len(signal_channel)) * signal_channel
                
                # Calculate FFT
                fft = np.abs(np.fft.rfft(windowed_signal))
                freqs = np.fft.rfftfreq(n_times, d=1/sfreq)
                
                # Extract power in each band
                band_powers = []
                for band_name, (fmin, fmax) in bands.items():
                    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                    if np.any(idx):
                        power = np.mean(fft[idx]**2)
                        band_powers.append(power)
                    else:
                        band_powers.append(0)
                
                # Calculate band ratios
                alpha_beta_ratio = band_powers[2] / (band_powers[3] + 1e-10)  # Alpha/Beta
                theta_beta_ratio = band_powers[1] / (band_powers[3] + 1e-10)  # Theta/Beta
                
                # Add all features for this channel
                sample_features.append(band_powers[0])  # delta power
                sample_features.append(band_powers[2])  # alpha power
                sample_features.append(alpha_beta_ratio)
                sample_features.append(theta_beta_ratio)
            
            features.append(sample_features)
        
        # Convert to array and combine with original features
        features_array = np.array(features)
        print(f"  • Feature extraction completed: {features_array.shape[1]} new features")
        
        # Normalize new features
        scaler = StandardScaler()
        features_array = scaler.fit_transform(features_array)
        
        # Combine with original features
        X_with_features = np.hstack([X, features_array])
        print(f"  • Final dimensions: {X_with_features.shape}")
        
        return X_with_features
    else:
        print("  • Using simplified feature set (original features only)")
        return X

def create_standard_pipelines():
    """
    Creates simplified standard pipelines (non-CSP models)
    """
    pipelines = {
        'PCA_SVM': Pipeline([
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ]),
        'PCA_RF': Pipeline([
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced'))
        ]),
        'PCA_MLP': Pipeline([
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', MLPClassifier(max_iter=2000, hidden_layer_sizes=(100, 50)))
        ]),
        'SMOTE_PCA_RF': ImbPipeline([
            ('sampling', SMOTE(random_state=42)),
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced'))
        ])
    }
    return pipelines

def create_csp_pipelines():
    """
    Creates simplified CSP-based pipelines
    """
    pipelines = {
        'CSP_SVM': Pipeline([
            ('csp', CSP(n_components=8, reg=None, log=True, cov_est='epoch')),
            ('scaler', StandardScaler()),
            ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ]),
        'CSP_RF': Pipeline([
            ('csp', CSP(n_components=8, reg=None, log=True, cov_est='epoch')),
            ('scaler', StandardScaler()),
            ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced'))
        ]),
        'CSP_Shrinkage_SVM': Pipeline([
            ('csp', CSP(n_components=8, reg='shrinkage', log=True, cov_est='epoch')),
            ('scaler', StandardScaler()),
            ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ])
    }
    return pipelines

def create_mixed_ensemble():
    """
    Creates a mixed ensemble combining CSP and non-CSP models
    """
    # Create standard pipeline
    standard_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95)),
        ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced'))
    ])
    
    # Create CSP pipeline
    csp_pipeline = Pipeline([
        ('csp', CSP(n_components=8, reg='ledoit_wolf', log=True, cov_est='epoch')),
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ])
    
    # Create custom ensemble
    ensemble = EnsembleClassifier(
        standard_pipeline=standard_pipeline,
        csp_pipeline=csp_pipeline,
        voting='soft'
    )
    
    return ensemble

def create_param_grids():
    """
    Creates simplified parameter grids for optimization
    """
    standard_param_grids = {
        'PCA_SVM': {
            'dimension_reduction__n_components': [0.9, 0.95, 0.99],
            'classifier__C': [0.1, 1, 10],
            'classifier__gamma': ['scale', 'auto']
        },
        'PCA_RF': {
            'dimension_reduction__n_components': [0.9, 0.95, 0.99],
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 20]
        },
        'PCA_MLP': {
            'dimension_reduction__n_components': [0.9, 0.95, 0.99],
            'classifier__hidden_layer_sizes': [(50,), (100,), (100, 50)],
            'classifier__alpha': [0.0001, 0.001]
        },
        'SMOTE_PCA_RF': {
            'sampling__k_neighbors': [5],
            'dimension_reduction__n_components': [0.9, 0.95],
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 20]
        }
    }
    
    csp_param_grids = {
        'CSP_SVM': {
            'csp__n_components': [4, 6, 8],
            'csp__log': [True],
            'classifier__C': [0.1, 1, 10]
        },
        'CSP_RF': {
            'csp__n_components': [4, 6, 8],
            'csp__log': [True],
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 20]
        },
        'CSP_Shrinkage_SVM': {
            'csp__n_components': [4, 6, 8],
            'classifier__C': [0.1, 1, 10]
        }
    }
    
    return standard_param_grids, csp_param_grids

def evaluate_pipeline(pipeline, param_grid, X, y, X_3d=None, name='Pipeline', scoring='accuracy', cv=None):
    """
    Evaluates a pipeline with cross-validation and hyperparameter optimization
    """
    print(f"\n🔍 Evaluating pipeline: {name}")
    
    if cv is None:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        verbose=1,
        error_score='raise',
        return_train_score=True
    )
    
    try:
        # Determine if it's a CSP pipeline
        is_csp_pipeline = 'CSP' in name or hasattr(pipeline, 'named_steps') and 'csp' in pipeline.named_steps
        
        # Use appropriate data based on pipeline type
        if is_csp_pipeline and X_3d is not None:
            print(f"  • Using 3D data format for CSP pipeline")
            grid_search.fit(X_3d.astype(np.float64), y)
        else:
            print(f"  • Using 2D data format for standard pipeline")
            grid_search.fit(X, y)
        
        # Save results
        result = {
            'best_params': grid_search.best_params_,
            'best_score': grid_search.best_score_,
            'cv_results': grid_search.cv_results_,
            'best_estimator': grid_search.best_estimator_,
            'is_csp': is_csp_pipeline
        }
        
        print(f"Best score for {name}: {result['best_score']:.3f} with {scoring}")
        print(f"Best parameters: {result['best_params']}")
        return result
    except Exception as e:
        print(f"❌ Error evaluating {name}: {str(e)}")
        return None

def evaluate_mixed_ensemble(ensemble, X, y, X_3d, name='Mixed_Ensemble', scoring='accuracy', cv=None):
    """
    Evaluates the mixed ensemble that combines CSP and non-CSP models
    """
    print(f"\n🔍 Evaluating mixed ensemble: {name}")
    
    if cv is None:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    try:
        # Train the full ensemble
        ensemble.fit(X, y, X_3d=X_3d)
        
        # Calculate score with cross-validation
        scores = []
        for fold_idx, (train_idx, test_idx) in enumerate(tqdm(cv.split(X, y), desc="CV Folds", total=cv.get_n_splits())):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # For CSP, we need 3D data
            if X_3d is not None:
                X_3d_train, X_3d_test = X_3d[train_idx], X_3d[test_idx]
            else:
                X_3d_train = X_3d_test = None
            
            # Train and predict
            ensemble.fit(X_train, y_train, X_3d=X_3d_train)
            y_pred = ensemble.predict(X_test, X_3d=X_3d_test)
            
            # Calculate score
            if scoring == 'accuracy':
                score = accuracy_score(y_test, y_pred)
            elif scoring == 'f1_weighted':
                score = f1_score(y_test, y_pred, average='weighted')
            elif scoring == 'roc_auc':
                y_pred_proba = ensemble.predict_proba(X_test, X_3d=X_3d_test)
                score = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
            else:
                score = accuracy_score(y_test, y_pred)
            
            scores.append(score)
            print(f"  • Fold {fold_idx+1}/{cv.get_n_splits()}: {score:.3f}")
        
        # Calculate average scores
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        
        result = {
            'scores': scores,
            'mean_score': mean_score,
            'std_score': std_score,
            'best_estimator': ensemble,
            'is_csp': True  # It's a mixed model that includes CSP
        }
        
        print(f"Average score for {name}: {mean_score:.3f} ± {std_score:.3f} with {scoring}")
        return result
    except Exception as e:
        print(f"❌ Error evaluating mixed ensemble {name}: {str(e)}")
        return None

def evaluate_all_pipelines(X, y, X_3d, scoring='accuracy'):
    """
    Evaluates all pipelines and returns the best one
    """
    # Create pipelines and parameter grids
    standard_pipelines = create_standard_pipelines()
    csp_pipelines = create_csp_pipelines()
    standard_param_grids, csp_param_grids = create_param_grids()
    
    results = {}
    best_score = 0
    best_pipeline = None
    best_pipeline_name = None
    
    # Create cross-validation object
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Evaluate standard pipelines
    for name, pipeline in standard_pipelines.items():
        result = evaluate_pipeline(
            pipeline, 
            standard_param_grids.get(name, {}),
            X, y, None,
            name=name,
            scoring=scoring,
            cv=cv
        )
        if result is not None:
            results[name] = result
            if result['best_score'] > best_score:
                best_score = result['best_score']
                best_pipeline = result['best_estimator']
                best_pipeline_name = name
    
    # Evaluate CSP pipelines
    if X_3d is not None:
        for name, pipeline in csp_pipelines.items():
            result = evaluate_pipeline(
                pipeline,
                csp_param_grids.get(name, {}),
                X, y, X_3d,
                name=name,
                scoring=scoring,
                cv=cv
            )
            if result is not None:
                results[name] = result
                if result['best_score'] > best_score:
                    best_score = result['best_score']
                    best_pipeline = result['best_estimator']
                    best_pipeline_name = name
    
    # Evaluate Mixed Ensemble
    if X_3d is not None:
        mixed_ensemble = create_mixed_ensemble()
        result = evaluate_mixed_ensemble(
            mixed_ensemble,
            X, y, X_3d,
            name='Mixed_Ensemble',
            scoring=scoring,
            cv=cv
        )
        if result is not None:
            results['Mixed_Ensemble'] = result
            if result['mean_score'] > best_score:
                best_score = result['mean_score']
                best_pipeline = result['best_estimator']
                best_pipeline_name = 'Mixed_Ensemble'
    
    evaluation_result = {
        'results': results,
        'best_pipeline': best_pipeline,
        'best_pipeline_name': best_pipeline_name,
        'best_score': best_score,
        'scoring_metric': scoring
    }
    
    return evaluation_result

def evaluate_with_multiple_metrics(X, y, X_3d):
    """
    Evaluates pipelines using multiple metrics
    """
    # Reduced to fewer metrics for simplicity
    metrics = ['accuracy', 'f1_weighted']
    all_results = {}
    
    for metric in metrics:
        print(f"\n\n📊 Evaluating with metric: {metric.upper()}")
        evaluation_result = evaluate_all_pipelines(X, y, X_3d, scoring=metric)
        all_results[metric] = evaluation_result
        
        print(f"\n✅ Best pipeline for {metric}: {evaluation_result['best_pipeline_name']}")
        print(f"   Score: {evaluation_result['best_score']:.3f}")
    
    # Find the pipeline with best overall performance
    best_overall_score = 0
    best_overall_pipeline = None
    best_overall_name = None
    best_metric = None
    
    for metric, result in all_results.items():
        if result['best_score'] > best_overall_score:
            best_overall_score = result['best_score']
            best_overall_pipeline = result['best_pipeline']
            best_overall_name = result['best_pipeline_name']
            best_metric = metric
    
    print(f"\n🏆 Best overall pipeline is {best_overall_name} with {best_metric}: {best_overall_score:.3f}")
    
    return {
        'all_metric_results': all_results,
        'best_overall_pipeline': best_overall_pipeline,
        'best_overall_name': best_overall_name,
        'best_overall_score': best_overall_score,
        'best_metric': best_metric
    }

def save_results(evaluation_result, timestamp=None):
    """
    Saves results and best pipeline
    """
    if timestamp is None:
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Create directory for this evaluation
    eval_dir = os.path.join(models_dir, f'evaluation_{timestamp}')
    os.makedirs(eval_dir, exist_ok=True)
    
    # Save all metric results if available
    if 'all_metric_results' in evaluation_result:
        all_results = evaluation_result['all_metric_results']
        
        for metric, metric_result in all_results.items():
            results_file = os.path.join(eval_dir, f'pipeline_evaluations_{metric}.json')
            
            with open(results_file, 'w') as f:
                serializable_results = {}
                for name, result in metric_result['results'].items():
                    if result is not None:
                        serializable_results[name] = {
                            'best_params': result.get('best_params', {}),
                            'best_score': float(result.get('best_score', result.get('mean_score', 0))),
                            'is_csp': result.get('is_csp', False)
                        }
                json.dump(serializable_results, f, indent=4)
        
        # Save best overall pipeline
        best_pipeline = evaluation_result['best_overall_pipeline']
        best_pipeline_name = evaluation_result['best_overall_name']
        best_score = evaluation_result['best_overall_score']
        best_metric = evaluation_result['best_metric']
        
        # Save using joblib
        best_pipeline_path = os.path.join(eval_dir, 'best_pipeline.joblib')
        dump(best_pipeline, best_pipeline_path)
        
        # Also save in main directory for easy access
        dump(best_pipeline, os.path.join(models_dir, 'best_pipeline.joblib'))
        
        # Save best pipeline info
        pipeline_info = {
            'best_pipeline_name': best_pipeline_name,
            'best_score': float(best_score),
            'best_metric': best_metric,
            'timestamp': timestamp,
            'pipeline_path': best_pipeline_path,
            'is_csp': best_pipeline_name.startswith('CSP') or best_pipeline_name == 'Mixed_Ensemble',
            'is_ensemble': 'Ensemble' in best_pipeline_name
        }
        
        with open(os.path.join(eval_dir, 'best_pipeline_info.json'), 'w') as f:
            json.dump(pipeline_info, f, indent=4)
        
        with open(os.path.join(models_dir, 'best_pipeline_info.json'), 'w') as f:
            json.dump(pipeline_info, f, indent=4)
    
    print(f"\n💾 Results saved in {eval_dir}")
    print(f"💾 Best pipeline saved as {os.path.join(models_dir, 'best_pipeline.joblib')}")
    
    return eval_dir

def simulate_real_time_prediction(pipeline, X, y, X_3d=None, chunk_size=10):
    """
    Simulates real-time predictions with performance metrics
    """
    predictions = []
    true_labels = []
    probabilities = []
    times = []
    
    # Determine if it's a CSP or mixed pipeline
    is_csp = (hasattr(pipeline, 'named_steps') and 'csp' in pipeline.named_steps) or \
             isinstance(pipeline, EnsembleClassifier)
    
    print("\nSimulating real-time prediction...")
    
    for i in tqdm(range(0, len(X), chunk_size), desc="Processing chunks"):
        X_chunk = X[i:i + chunk_size]
        y_chunk = y[i:i + chunk_size]
        
        # Prepare 3D data for CSP if needed
        X_3d_chunk = None
        if is_csp and X_3d is not None:
            X_3d_chunk = X_3d[i:i + chunk_size]
        
        start_time = time.time()
        
        if isinstance(pipeline, EnsembleClassifier) and X_3d_chunk is not None:
            y_pred = pipeline.predict(X_chunk, X_3d_chunk)
            if hasattr(pipeline, 'predict_proba'):
                try:
                    y_pred_proba = pipeline.predict_proba(X_chunk, X_3d_chunk)
                    probabilities.extend(y_pred_proba)
                except:
                    pass
        elif is_csp and X_3d_chunk is not None:
            y_pred = pipeline.predict(X_3d_chunk)
            if hasattr(pipeline, 'predict_proba'):
                try:
                    y_pred_proba = pipeline.predict_proba(X_3d_chunk)
                    probabilities.extend(y_pred_proba)
                except:
                    pass
        else:
            y_pred = pipeline.predict(X_chunk)
            if hasattr(pipeline, 'predict_proba'):
                try:
                    y_pred_proba = pipeline.predict_proba(X_chunk)
                    probabilities.extend(y_pred_proba)
                except:
                    pass
                    
        pred_time = time.time() - start_time
        times.append(pred_time)
        
        predictions.extend(y_pred.tolist())
        true_labels.extend(y_chunk.tolist())
        
        # Calculate current accuracy for feedback
        current_acc = accuracy_score(true_labels, predictions)
        
    # Calculate final metrics
    final_metrics = {
        'accuracy': accuracy_score(true_labels, predictions),
        'f1': f1_score(true_labels, predictions, average='weighted'),
        'precision': precision_score(true_labels, predictions, average='weighted'),
        'recall': recall_score(true_labels, predictions, average='weighted'),
        'avg_prediction_time': np.mean(times),
        'max_prediction_time': np.max(times),
        'min_prediction_time': np.min(times)
    }
    
    if probabilities and len(np.unique(true_labels)) > 1:
        try:
            final_metrics['roc_auc'] = roc_auc_score(true_labels, probabilities, multi_class='ovr')
        except:
            print("Could not calculate ROC AUC (possibly only one class)")
    
    print("\n📊 Final metrics from simulation:")
    for metric_name, metric_value in final_metrics.items():
        print(f"  • {metric_name}: {metric_value:.4f}")
        
    return predictions, true_labels, probabilities, times, final_metrics

def main():
    """
    Main function to run the entire EEG classification pipeline
    """
    # Load and prepare data
    print("📊 Loading data...")
    
    # Set paths - modify these paths to match your data location
    data_dir = '../data'
    models_dir = '../models'
    os.makedirs(models_dir, exist_ok=True)
    
    # Load data - modify these filenames as needed
    X = np.load(os.path.join(models_dir, 'X_preprocessed.npy'))
    y = np.load(os.path.join(models_dir, 'y_labels.npy'))
    print(f"✅ Data loaded: {X.shape} samples, {len(np.unique(y))} classes")

    # Load preprocessing information
    try:
        with open(os.path.join(models_dir, 'preprocessing_info.json'), 'r') as f:
            preprocess_info = json.load(f)
        
        # Get dimensions
        n_epochs = len(X)
        n_channels = len(preprocess_info['channels'])
        n_times = X.shape[1] // n_channels
        sfreq = preprocess_info.get('sampling_frequency', 160)
        
        print(f"  • EEG channels: {n_channels}")
        print(f"  • Time points: {n_times}")
        print(f"  • Sampling frequency: {sfreq} Hz")
        
        # Reshape for CSP and ensure correct data type
        X_3d = X.reshape(n_epochs, n_channels, n_times).astype(np.float64)
        print(f"  • 3D data for CSP: {X_3d.shape}")
    except Exception as e:
        print(f"⚠️ Error getting preprocessing information: {str(e)}")
        # If failed to get info, try to infer from data shape
        try:
            n_epochs = len(X)
            n_channels = 64  # Assume 64 channels
            n_times = X.shape[1] // n_channels
            X_3d = X.reshape(n_epochs, n_channels, n_times).astype(np.float64)
            print(f"  • 3D data for CSP (inferred): {X_3d.shape}")
        except:
            print("  • Could not create 3D data for CSP")
            X_3d = None
    
    # Augment data (limited augmentation for speed)
    X_aug, y_aug, X_3d_aug = augment_eeg_data(X, y, X_3d, n_augmentations=1)
    
    # Extract additional features (simplified)
    X_enhanced = extract_eeg_features(X_aug, X_3d_aug, sfreq=sfreq, simplified=True)
    
    # Timestamp for this session
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Evaluate pipelines with multiple metrics
    print("\n🚀 Starting pipeline evaluation with multiple metrics...")
    multi_metric_results = evaluate_with_multiple_metrics(X_enhanced, y_aug, X_3d_aug)
    
    # Save results
    eval_dir = save_results(multi_metric_results, timestamp)
    
    # Test the best pipeline with real-time simulation
    best_pipeline = multi_metric_results['best_overall_pipeline']
    best_pipeline_name = multi_metric_results['best_overall_name']
    
    if best_pipeline is not None:
        print("\n🚀 Testing the best pipeline in real-time simulation:")
        # Determine if the best pipeline is CSP or mixed
        is_csp_pipeline = best_pipeline_name.startswith('CSP') or best_pipeline_name == 'Mixed_Ensemble'
        
        # Use only a subset for demonstration
        test_samples = min(100, len(X_enhanced))
        predictions, true_labels, probabilities, times, metrics = simulate_real_time_prediction(
            best_pipeline, 
            X_enhanced[:test_samples], 
            y_aug[:test_samples],
            X_3d_aug[:test_samples] if X_3d_aug is not None else None
        )
        
        # Save simulation results
        with open(os.path.join(eval_dir, 'simulation_metrics.json'), 'w') as f:
            json.dump({k: float(v) for k, v in metrics.items()}, f, indent=4)
        
        print(f"\n✅ Evaluation complete. The best pipeline ({best_pipeline_name}) is ready to use.")
        print(f"   You can load this pipeline in other scripts using:")
        print(f"   pipeline = load_best_pipeline('{os.path.join(models_dir, 'best_pipeline.joblib')}')")
    else:
        print("\n❌ No optimal pipeline found. Please check the logs for more details.")

# Entry point
if __name__ == "__main__":
    main()
